# ML-09 — Validation Audit & Research Methodology Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faith-amanze/assignment1/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two findings from the FlyRank paper, and my methodology questions

Source: `docs/flyrank-seo-research-march-2026.pdf`, ML Appendix + Methodology pages. Questions
in the spirit of `writing-honest-claims`: where does the label come from, and does the
validation design actually carry the claim?

**Finding A — "What Predicts Health?" (Random Forest, feature importance for Health Score).**
The paper's own text flags this: *Health Score is itself built from Impressions (30 pts) +
Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts)*, and the model's top features are
exactly those four (Position 43%, Impressions 32%, Scroll Depth 15%, CTR 8%). My question: if
the label is a weighted sum of four inputs, and the model's top features by importance ARE
those same four inputs, how much of this "prediction" is the model discovering a real pattern
versus the model re-deriving the label's own arithmetic? The paper is careful to call this
"descriptive, not causal" in the text — but a cleaner test would be the same move I made in my
own leakage check: fit the model once with the four scoring inputs, once without, and show
how much of the reported "predictive power" survives their removal. Right now the reader has
to take the caveat on faith rather than see the number drop.

**Finding B — "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy).**
Two questions. First, the Methodology page states the Random Forest, Logistic Regression, and
Decision Tree all used an "80/20 split" with no mention of grouping by brand — and the ML
sample spans 57 brands. If the same brand's pages can land in both the train and test sides
(the same trap I built a client-holdout split to avoid in my own w05 model), part of that 71%
could be the model learning brand-specific quirks rather than a generalizable growth signal.
Second, 71% accuracy is reported with no base rate alongside it anywhere I found in the paper —
if, say, 60% of the sample is already "declining," a model could beat that number just by
leaning toward the majority class. Neither point means the finding is wrong; both are exactly
the kind of check `writing-honest-claims` asks for before trusting a headline number.

In [1]:
# ── Two findings from the FlyRank paper, pulled out as data for reference ──
# (paraphrased from docs/flyrank-seo-research-march-2026.pdf, ML Appendix + Methodology pages)
paper_findings = {
    "finding_A_health_score_rf": {
        "claim": "Random Forest feature importance for Health Score: Average Position 43%, "
                 "Impressions 32%, Scroll Depth 15%, CTR 8% (holdout-tested).",
        "label_formula": "Health Score = Impressions(30 pts) + Position(30 pts) + CTR(20 pts) + Scroll Depth(20 pts)",
        "split": "80/20, method unspecified (no grouping mentioned in Methodology page)",
        "n": "61,790 (local active-content feature-vector sample)",
    },
    "finding_B_growth_logreg": {
        "claim": "Logistic Regression, 71% holdout accuracy, separating growing vs declining pages. "
                 "Content Age is the strongest negative signal; Days Visible and recent Impressions "
                 "are the strongest positive signals.",
        "split": "80/20, method unspecified (no grouping by brand mentioned)",
        "n": "61,790 (same ML sample, 57 brands)",
        "base_rate_reported": False,
    },
}
for k, v in paper_findings.items():
    print(k)
    for kk, vv in v.items():
        print(f"    {kk}: {vv}")
    print()


finding_A_health_score_rf
    claim: Random Forest feature importance for Health Score: Average Position 43%, Impressions 32%, Scroll Depth 15%, CTR 8% (holdout-tested).
    label_formula: Health Score = Impressions(30 pts) + Position(30 pts) + CTR(20 pts) + Scroll Depth(20 pts)
    split: 80/20, method unspecified (no grouping mentioned in Methodology page)
    n: 61,790 (local active-content feature-vector sample)

finding_B_growth_logreg
    claim: Logistic Regression, 71% holdout accuracy, separating growing vs declining pages. Content Age is the strongest negative signal; Days Visible and recent Impressions are the strongest positive signals.
    split: 80/20, method unspecified (no grouping by brand mentioned)
    n: 61,790 (same ML sample, 57 brands)
    base_rate_reported: False



## 2. My Week-5 model under an honest split, before/after

Direct test of the concern raised in Finding B above, run on my own data: what happens to my
w05 model's numbers if I swap my grouped client-holdout split for a plain random 80/20 split —
the same split style the paper's methodology page describes, with no mention of grouping?

In [2]:
# ── My w05 model, re-run under a RANDOM split (like the paper's unspecified 80/20)
#    vs the GROUPED client-holdout split I actually used. Same feature set, same target. ──
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score

RANDOM_STATE = 42
_candidates = [
    "/workspaces/assignment1/data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
    "data/raw/content_refresh_anonymized.csv",
]
_data_path = next(p for p in _candidates if os.path.exists(p))
df = pd.read_csv(_data_path)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

numeric_features = [
    "avg_position", "impressions_90d", "clicks_90d", "ctr",
    "pageviews_90d", "sessions_90d", "users_90d", "engaged_sessions_90d",
    "ai_sessions_90d", "scroll_events_90d", "days_with_impressions",
    "days_with_sessions", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "word_count", "char_count", "content_age_days", "days_since_last_update",
    "search_volume", "competition", "cpc",
]
categorical_features = ["content_type", "main_intent", "competition_level"]
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["has_position_data"] = (df["avg_position"] > 0).astype(int)
numeric_features += ["has_keyword_data", "has_word_count", "has_position_data"]

feature_cols = numeric_features + categorical_features
X = df[feature_cols].copy()
y = df["is_declining_label"].copy()
groups = df["client_id"]

def make_pipe():
    prep = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_features),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="unknown")),
                           ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
    ])
    return Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))])

def precision_at_k(y_true, scores, k=50):
    order = np.argsort(-scores)[:k]
    return y_true.values[order].mean()

def evaluate(X_train, X_test, y_train, y_test, label):
    pipe = make_pipe()
    pipe.fit(X_train, y_train)
    scores = pipe.predict_proba(X_test)[:, 1]
    preds = pipe.predict(X_test)
    return {
        "split": label,
        "n_test": len(X_test),
        "base_rate": round(y_test.mean(), 3),
        "accuracy": round(accuracy_score(y_test, preds), 3),
        "roc_auc": round(roc_auc_score(y_test, scores), 3),
        "precision@50": round(precision_at_k(y_test, scores, 50), 3),
    }

# (A) RANDOM 80/20 row split -- mirrors the paper's unspecified split, same client can appear
# in both train and test since nothing groups by client_id/brand.
Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
random_result = evaluate(Xtr_r, Xte_r, ytr_r, yte_r, "random_80_20 (paper-style)")

# (B) GROUPED client-holdout split -- what I actually used in w04/w05.
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups))
grouped_result = evaluate(X.iloc[train_idx], X.iloc[test_idx], y.iloc[train_idx], y.iloc[test_idx], "grouped_client_holdout (mine)")

before_after = pd.DataFrame([random_result, grouped_result])
print(before_after.to_string(index=False))
print(f"\nAccuracy gap (random - grouped): {random_result['accuracy'] - grouped_result['accuracy']:.3f}")
print(f"ROC AUC gap (random - grouped):  {random_result['roc_auc'] - grouped_result['roc_auc']:.3f}")


                        split  n_test  base_rate  accuracy  roc_auc  precision@50
   random_80_20 (paper-style)    6000      0.542     0.643    0.695          0.82
grouped_client_holdout (mine)    7115      0.517     0.563    0.600          0.74

Accuracy gap (random - grouped): 0.080
ROC AUC gap (random - grouped):  0.095


### What the honest-split test shows

A random 80/20 split (no grouping) inflates accuracy by **+8.0 points** (0.643 vs 0.563) and
ROC AUC by **+9.5 points** (0.695 vs 0.600) compared to my grouped client-holdout split, on the
exact same model and feature set. That's not a small rounding difference — it's the same shape
of inflation the paper's unspecified split could be producing for its 71% figure, since 57
brands sharing an 80/20 split (if not grouped) creates exactly this opportunity. This doesn't
mean the paper's number is wrong; it means the number can't be fully trusted without knowing
whether grouping was used, and this test shows concretely why that detail matters.

## 3. Leakage audit on my final feature set

Same "attack your own model" hunt from `w03_feature_leakage_check.ipynb` — re-run here on the
exact feature set that went into `w05_model.ipynb`, to confirm nothing leaked back in since then.

In [3]:
# ── Leakage audit, same hunt as w03_feature_leakage_check.ipynb, on my FINAL feature set ──
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score as _auc

def make_rf_pipe(num_cols, cat_cols):
    prep = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="unknown")),
                           ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
    ])
    return Pipeline([("prep", prep),
                      ("clf", RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=20,
                                                      class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1))])

def fit_auc(num_cols, cat_cols):
    Xf = df[num_cols + cat_cols]
    Xtr, Xte = Xf.iloc[train_idx], Xf.iloc[test_idx]
    ytr, yte = y.iloc[train_idx], y.iloc[test_idx]
    pipe = make_rf_pipe(num_cols, cat_cols)
    pipe.fit(Xtr, ytr)
    return _auc(yte, pipe.predict_proba(Xte)[:, 1])

honest_auc = fit_auc(numeric_features, categorical_features)
auc_with_trend_pct = fit_auc(numeric_features + ["trend_pct"], categorical_features)
suspect_family = ["impressions_last_30d", "impressions_prev_30d",
                   "clicks_last_30d", "clicks_prev_30d",
                   "sessions_last_30d", "sessions_prev_30d"]
auc_with_windows = fit_auc(numeric_features + suspect_family, categorical_features)

print(f"Final feature set (honest, grouped split): ROC AUC = {honest_auc:.3f}")
print(f"+ trend_pct added back:                    ROC AUC = {auc_with_trend_pct:.3f}  <- confession")
print(f"+ last30/prev30 window family added back:  ROC AUC = {auc_with_windows:.3f}  <- smaller leak")
print("\nSame result as the Week-3 leakage check, re-confirmed on the exact feature set that went")
print("into w05_model.ipynb -- no new leaks introduced since then.")


Final feature set (honest, grouped split): ROC AUC = 0.608
+ trend_pct added back:                    ROC AUC = 0.999  <- confession
+ last30/prev30 window family added back:  ROC AUC = 0.756  <- smaller leak

Same result as the Week-3 leakage check, re-confirmed on the exact feature set that went
into w05_model.ipynb -- no new leaks introduced since then.


## 4. Claim rewrite

My boldest sentence from `w05_model.ipynb`'s self-check, checked against the claim ladder and
rewritten to match what the evidence actually carries.

In [4]:
# ── Claim rewrite: my boldest sentence, before and after ──
claim_before = ("All three learned models beat the Week-4 rule at Precision@50 -- "
                 "logistic regression hit 0.740 vs the baseline's 0.480.")

claim_after = ("On this held-out set of 8 client accounts the model never trained on, "
                "ranking pages by predicted decline probability and reviewing the top 50 "
                "would have caught 74% true decliners, versus 48% from the current rule-based "
                "queue (base rate 52%) -- a decision-support signal for prioritizing the review "
                "queue, not a guarantee for any single page or client outside this sample.")

print("BEFORE (claim ladder check: implies a general, almost causal 'beats' without conditions)")
print(f"  {claim_before}\n")
print("AFTER (matches the evidence: observed on THIS holdout, states n, states it's ranking/decision-")
print("support not causal, names the base rate next to the headline number)")
print(f"  {claim_after}")


BEFORE (claim ladder check: implies a general, almost causal 'beats' without conditions)
  All three learned models beat the Week-4 rule at Precision@50 -- logistic regression hit 0.740 vs the baseline's 0.480.

AFTER (matches the evidence: observed on THIS holdout, states n, states it's ranking/decision-
support not causal, names the base rate next to the headline number)
  On this held-out set of 8 client accounts the model never trained on, ranking pages by predicted decline probability and reviewing the top 50 would have caught 74% true decliners, versus 48% from the current rule-based queue (base rate 52%) -- a decision-support signal for prioritizing the review queue, not a guarantee for any single page or client outside this sample.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
